# llmreport quickstart

Run a behavior and performance report on any Hugging Face causal language model. This notebook uses `gpt2` (small enough for a free CPU runtime).

In [ ]:
!pip install -q llmreport

## 1. Load a model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import llmreport

name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

## 2. Run the report

Quick mode takes a few minutes on CPU. Expand any section to see the numbers, caveats and the actual model outputs.

In [ ]:
report = llmreport.analyze(model, tokenizer)
report

## 3. Dig into the results

In [ ]:
report.metrics

In [ ]:
# Sample generations from the repetition check
for s in report["repetition"].details["samples"][:3]:
    print(s["prompt"], "->", s["output"][:120], "\n")

## 4. Export

In [ ]:
report.to_html("gpt2_report.html")
report.to_markdown("MODEL_CARD.md")
report.to_json("gpt2_report.json")

## 5. Compare two runs

For example, the same model on your own domain text versus the built-in texts, or a base model versus your fine-tuned version.

In [ ]:
domain = llmreport.analyze(
    model, tokenizer,
    checks=["perplexity", "repetition"],
    prompts={"perplexity": ["def add(a, b):\n    return a + b\n\nprint(add(2, 3))"]},
)
report.compare(domain, names=("built-in texts", "code"))

## 6. Add your own check

In [ ]:
from llmreport._utils import generate

@llmreport.register_analyzer
class AnswerLength(llmreport.Analyzer):
    name = "answer_length"
    title = "Answer length"
    description = "Average answer length in words."

    def run(self, model, tokenizer, config):
        prompts = ["What is AI?", "Describe a cat."]
        lengths = [len(generate(model, tokenizer, p, 48).split()) for p in prompts]
        avg = sum(lengths) / len(lengths)
        return self.result(f"Answers average {avg:.0f} words.", metrics={"avg_words": avg})

llmreport.analyze(model, tokenizer, checks=["answer_length"])